In [ ]:
import os
os.environ["NUMBA_ENABLE_CUDASIM"] = "1"

import warnings
from numba.core.errors import NumbaWarning, NumbaDeprecationWarning, NumbaPendingDeprecationWarning
import numpy as np
import matplotlib.pyplot as plt
from cpu_reference import black_scholes_call, monte_carlo_option_cpu, monte_carlo_pi_cpu
from gpu_kernels import run_monte_carlo_option_gpu

warnings.simplefilter("ignore", category=NumbaWarning)
warnings.simplefilter("ignore", category=NumbaDeprecationWarning)
warnings.simplefilter("ignore", category=NumbaPendingDeprecationWarning)
warnings.simplefilter("ignore", category=RuntimeWarning)

# 1. Paramètres de l'option financière
S0 = 100.0    # Prix sous-jacent
K = 100.0     # Prix d'exercice (Strike)
T = 1.0       # Échéance (1 ans)
r = 0.05      # Taux sans risque (5%)
sigma = 0.20  # Volatilité (20%)

# 2. Vérité terrain analytique
exact_price = black_scholes_call(S0, K, T, r, sigma)
print(f"--- Prix exact Black-Scholes : {exact_price:.4f} € ---")

# 3. Validation de la convergence Monte-Carlo CPU vs N (Erreur en 1/sqrt(N))
tirages = [10**2, 10**3, 10**4, 10**5, 10**6]
erreurs = []

for N in tirages:
    mc_price, std_err = monte_carlo_option_cpu(S0, K, T, r, sigma, N)
    erreur_absolue = abs(mc_price - exact_price)
    erreurs.append(erreur_absolue)
    print(f"N = {N:7d} | Prix MC CPU = {mc_price:.4f} € | Erreur = {erreur_absolue:.4f} €")

# 4. Test rapide du noyau GPU simulé
gpu_price = run_monte_carlo_option_gpu(S0, K, T, r, sigma, N_total=100000)
print(f"\nPrix estimé sur GPU (Simulateur) N=100k : {gpu_price:.4f} €")

--- Prix exact Black-Scholes : 10.4506 € ---
N =     100 | Prix MC CPU = 8.4746 € | Erreur = 1.9760 €
N =    1000 | Prix MC CPU = 10.7595 € | Erreur = 0.3089 €
N =   10000 | Prix MC CPU = 10.2313 € | Erreur = 0.2193 €
N =  100000 | Prix MC CPU = 10.4603 € | Erreur = 0.0097 €
N = 1000000 | Prix MC CPU = 10.4372 € | Erreur = 0.0134 €

Prix estimé sur GPU (Simulateur) N=100k : 10.4502 €
